# LumenY — 02: Feature Engineering

Builds the full feature matrix from processed OHLCV data across all timeframes.

**Architecture:** 1H as base index, features computed from 5m, 15m, 1H, 4H, 1D timeframes and merged into one row per timestamp.

**Output:** One feature parquet per pair + one combined parquet for all pairs ready for model training.

In [ ]:
# !pip install pandas numpy pandas-ta scikit-learn pyarrow tqdm matplotlib

In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('../backend/data/processed')
FEATURES_DIR  = Path('../backend/data/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD']

print('Ready.')
print(f'Features will be saved to: {FEATURES_DIR.resolve()}')

## 1. Feature Engineering Functions

Each function takes a OHLCV DataFrame and returns a DataFrame of features with a `_{tf}` suffix on column names so we know which timeframe each feature came from.

In [ ]:
def compute_features(df: pd.DataFrame, tf: str) -> pd.DataFrame:
    """
    Compute all TA features for a given OHLCV DataFrame.
    Returns a DataFrame with all features, columns suffixed with _{tf}.
    All features use only past data — no lookahead.
    """
    feat = pd.DataFrame(index=df.index)
    o, h, l, c = df['open'], df['high'], df['low'], df['close']
    v = df['volume'] if 'volume' in df.columns else pd.Series(0, index=df.index)
    
    # ── RETURNS & MOMENTUM ──────────────────────────────────────────
    
    # Log returns at multiple lookbacks
    for n in [1, 3, 6, 12, 24, 48]:
        feat[f'log_ret_{n}'] = np.log(c / c.shift(n))
    
    # RSI
    feat['rsi_14'] = ta.rsi(c, length=14)
    feat['rsi_28'] = ta.rsi(c, length=28)
    
    # RSI slope — is momentum accelerating or decelerating?
    feat['rsi_slope'] = feat['rsi_14'] - feat['rsi_14'].shift(3)
    
    # RSI divergence flag — price makes new high but RSI doesn't (or vice versa)
    price_higher = (c > c.shift(5)).astype(int)
    rsi_higher   = (feat['rsi_14'] > feat['rsi_14'].shift(5)).astype(int)
    feat['rsi_divergence'] = (price_higher != rsi_higher).astype(int)
    
    # MACD
    macd = ta.macd(c, fast=12, slow=26, signal=9)
    if macd is not None:
        feat['macd']        = macd.iloc[:, 0]
        feat['macd_signal'] = macd.iloc[:, 2]
        feat['macd_hist']   = macd.iloc[:, 1]
    
    # Distance from moving averages (normalized by price)
    for n in [20, 50, 200]:
        ma = ta.sma(c, length=n)
        feat[f'dist_ma_{n}'] = (c - ma) / c
    
    # EMA cross signal
    ema_fast = ta.ema(c, length=9)
    ema_slow = ta.ema(c, length=21)
    feat['ema_cross'] = (ema_fast > ema_slow).astype(int)
    feat['ema_dist']  = (ema_fast - ema_slow) / c

    # ── VOLATILITY ──────────────────────────────────────────────────
    
    # ATR (normalized by price)
    atr_14 = ta.atr(h, l, c, length=14)
    atr_28 = ta.atr(h, l, c, length=28)
    feat['atr_14_norm'] = atr_14 / c
    feat['atr_28_norm'] = atr_28 / c
    
    # ATR ratio — current vs longer-term (compression/expansion)
    feat['atr_ratio'] = atr_14 / atr_28
    
    # Rolling std of log returns
    log_ret = np.log(c / c.shift(1))
    feat['rvol_12']  = log_ret.rolling(12).std()
    feat['rvol_24']  = log_ret.rolling(24).std()
    feat['rvol_48']  = log_ret.rolling(48).std()
    
    # Realized vol ratio — short vs long (are we in a vol spike?)
    feat['rvol_ratio'] = feat['rvol_12'] / feat['rvol_48']
    
    # High-low range normalized
    feat['hl_range'] = (h - l) / c
    
    # Bollinger Band width and position
    bb = ta.bbands(c, length=20, std=2)
    if bb is not None:
        bb_upper = bb.iloc[:, 0]
        bb_mid   = bb.iloc[:, 1]
        bb_lower = bb.iloc[:, 2]
        feat['bb_width']    = (bb_upper - bb_lower) / bb_mid
        feat['bb_position'] = (c - bb_lower) / (bb_upper - bb_lower + 1e-10)
    
    # ── MARKET STRUCTURE & KEY LEVELS ───────────────────────────────
    
    # Distance from N-bar high/low (normalized)
    for n in [20, 50]:
        feat[f'dist_high_{n}'] = (c - h.rolling(n).max()) / c
        feat[f'dist_low_{n}']  = (c - l.rolling(n).min()) / c
    
    # Breakout flag — price above N-bar high
    feat['breakout_20'] = (c > h.shift(1).rolling(20).max()).astype(int)
    feat['breakdown_20'] = (c < l.shift(1).rolling(20).min()).astype(int)
    
    # Trend slope — linear regression slope of close over N bars
    def rolling_slope(series, n):
        slopes = series.copy() * np.nan
        x = np.arange(n)
        for i in range(n, len(series)):
            y = series.iloc[i-n:i].values
            if not np.any(np.isnan(y)):
                slopes.iloc[i] = np.polyfit(x, y, 1)[0] / series.iloc[i]
        return slopes
    
    feat['trend_slope_20'] = rolling_slope(c, 20)
    
    # Candle body and wick ratios
    body  = abs(c - o)
    range_ = h - l + 1e-10
    feat['body_ratio']       = body / range_
    feat['upper_wick_ratio'] = (h - pd.concat([c, o], axis=1).max(axis=1)) / range_
    feat['lower_wick_ratio'] = (pd.concat([c, o], axis=1).min(axis=1) - l) / range_
    
    # ADX — trend strength
    adx = ta.adx(h, l, c, length=14)
    if adx is not None:
        feat['adx'] = adx.iloc[:, 0]
    
    # ── NEW: BUYING/SELLING PRESSURE ────────────────────────────────
    
    # Close position within bar (0=low, 1=high) — proxy for buying pressure
    close_pos = (c - l) / (h - l + 1e-10)
    feat['close_position'] = close_pos
    
    # Net pressure: +1 = close at high (buyers won), -1 = close at low (sellers won)
    feat['net_pressure'] = 2 * close_pos - 1
    
    # Rolling net pressure — sustained buying/selling over N bars
    feat['net_pressure_6'] = feat['net_pressure'].rolling(6).mean()
    feat['net_pressure_12'] = feat['net_pressure'].rolling(12).mean()
    
    # ── NEW: Z-SCORE MEAN REVERSION ─────────────────────────────────
    
    sma_20 = ta.sma(c, length=20)
    std_20 = c.rolling(20).std()
    sma_50 = ta.sma(c, length=50)
    std_50 = c.rolling(50).std()
    
    feat['zscore_20'] = (c - sma_20) / (std_20 + 1e-10)
    feat['zscore_50'] = (c - sma_50) / (std_50 + 1e-10)
    feat['zscore_change_3'] = feat['zscore_20'] - feat['zscore_20'].shift(3)
    feat['zscore_extreme'] = (feat['zscore_20'].abs() > 2).astype(int)
    
    # ── NEW: VOLATILITY REGIME ──────────────────────────────────────
    
    # Percentile rank of current vol ratio over trailing 100 bars
    feat['vol_regime'] = feat['rvol_ratio'].rolling(100).rank(pct=True)
    
    # Rate of change in realized vol
    feat['vol_change_6'] = feat['rvol_12'] - feat['rvol_12'].shift(6)
    
    # ATR z-score — how unusual is current volatility?
    atr_mean_50 = atr_14.rolling(50).mean() if atr_14 is not None else None
    atr_std_50 = atr_14.rolling(50).std() if atr_14 is not None else None
    if atr_mean_50 is not None:
        feat['atr_zscore'] = (atr_14 - atr_mean_50) / (atr_std_50 + 1e-10)
    
    # Current bar range vs recent average — detects sudden range expansion
    feat['range_ratio_3'] = feat['hl_range'] / (feat['hl_range'].rolling(3).mean() + 1e-10)
    
    # ── NEW: ENHANCED MOMENTUM ──────────────────────────────────────
    
    # Momentum acceleration (second derivative of price)
    feat['roc_acceleration'] = log_ret - log_ret.shift(1)
    
    # Consecutive direction count
    direction = (log_ret > 0).astype(int) * 2 - 1  # +1 or -1
    consec = direction.copy() * 0.0
    for i in range(1, len(consec)):
        if direction.iloc[i] == direction.iloc[i-1]:
            consec.iloc[i] = consec.iloc[i-1] + direction.iloc[i]
        else:
            consec.iloc[i] = direction.iloc[i]
    feat['consecutive_dir'] = consec
    
    # Return skewness — fat tail detection
    feat['ret_skew_12'] = log_ret.rolling(12).skew()
    feat['ret_skew_24'] = log_ret.rolling(24).skew()
    
    # ── NEW: VOLUME MICROSTRUCTURE ──────────────────────────────────
    
    # Relative volume vs 20-bar average
    vol_sma_20 = v.rolling(20).mean()
    vol_std_20 = v.rolling(20).std()
    feat['vol_sma_ratio'] = v / (vol_sma_20 + 1e-10)
    feat['vol_zscore'] = (v - vol_sma_20) / (vol_std_20 + 1e-10)
    
    # Volume trend — short vs long MA
    feat['vol_trend'] = v.rolling(5).mean() / (v.rolling(20).mean() + 1e-10)
    
    # Price-volume correlation
    feat['pv_corr_12'] = log_ret.rolling(12).corr(v)
    
    # Order flow imbalance proxy
    ofi = (2 * close_pos - 1) * v
    feat['ofi_6'] = ofi.rolling(6).sum()
    feat['ofi_12'] = ofi.rolling(12).sum()
    
    # Add timeframe suffix to all columns
    feat.columns = [f'{col}_{tf}' for col in feat.columns]
    
    return feat


print(f'Feature function ready.')

In [ ]:
def compute_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute time-based features from the index.
    These are the same regardless of timeframe.
    """
    feat = pd.DataFrame(index=df.index)
    
    # Hour of day — FX session patterns
    feat['hour'] = df.index.hour
    
    # Session flags
    feat['session_asian']  = ((df.index.hour >= 0)  & (df.index.hour < 8)).astype(int)
    feat['session_london'] = ((df.index.hour >= 8)  & (df.index.hour < 16)).astype(int)
    feat['session_ny']     = ((df.index.hour >= 13) & (df.index.hour < 21)).astype(int)
    feat['session_overlap'] = ((df.index.hour >= 13) & (df.index.hour < 16)).astype(int)  # London/NY overlap — most volatile
    
    # Day of week
    feat['day_of_week'] = df.index.dayofweek  # 0=Mon, 4=Fri
    feat['is_monday']   = (df.index.dayofweek == 0).astype(int)
    feat['is_friday']   = (df.index.dayofweek == 4).astype(int)
    
    # Month — some FX seasonality exists
    feat['month'] = df.index.month
    
    return feat


print('Time feature function ready.')

In [ ]:
def compute_session_features(df_1h: pd.DataFrame) -> pd.DataFrame:
    """
    Compute FX session-relative features from 1H OHLCV data.
    Sessions: Asian (00-08 UTC), London (08-16 UTC), NY (13-21 UTC).
    
    IMPORTANT: Session open bars use shift(1) — at row T=08:00, you see the
    PREVIOUS London open (yesterday's 08:00 bar), not today's which hasn't closed.
    Today's London open becomes visible at T=09:00.
    """
    feat = pd.DataFrame(index=df_1h.index)
    c = df_1h['close']
    h = df_1h['high']
    l = df_1h['low']
    
    hours = df_1h.index.hour
    
    # ── London Open Range ──────────────────────────────────────────
    # The 08:00 UTC bar = first hour of London session
    # shift(1) so the 08:00 bar's data is only available at 09:00
    london_open_mask = (hours == 8)
    london_open_high = h.where(london_open_mask).shift(1).ffill()
    london_open_low = l.where(london_open_mask).shift(1).ffill()
    london_open_price = df_1h['open'].where(london_open_mask).shift(1).ffill()
    
    feat['dist_london_high'] = (c - london_open_high) / c
    feat['dist_london_low'] = (c - london_open_low) / c
    feat['dist_london_open'] = (c - london_open_price) / c
    
    # ── NY Open Range ──────────────────────────────────────────────
    # shift(1) so the 13:00 bar's data is only available at 14:00
    ny_open_mask = (hours == 13)
    ny_open_high = h.where(ny_open_mask).shift(1).ffill()
    ny_open_low = l.where(ny_open_mask).shift(1).ffill()
    ny_open_price = df_1h['open'].where(ny_open_mask).shift(1).ffill()
    
    feat['dist_ny_high'] = (c - ny_open_high) / c
    feat['dist_ny_low'] = (c - ny_open_low) / c
    feat['dist_ny_open'] = (c - ny_open_price) / c
    
    # ── Asian Session Range ────────────────────────────────────────
    # Asian session = 00:00-07:59 UTC.
    # Use shift(1) on the completed range so it's only visible from 09:00.
    asian_mask = (hours >= 0) & (hours < 8)
    
    # Get last Asian bar (07:00) high/low to mark end of Asian session
    # The full Asian range uses the previous day's completed Asian session
    dates = df_1h.index.date
    
    # Compute Asian high/low per date (only from Asian hours)
    asian_h = h.where(asian_mask)
    asian_l = l.where(asian_mask)
    daily_asian_high = asian_h.groupby(dates).transform('max')
    daily_asian_low = asian_l.groupby(dates).transform('min')
    
    # Only reveal at 08:00 (first non-Asian bar), then shift(1) for safety
    asian_complete_mask = (hours == 8)
    asian_high_final = daily_asian_high.where(asian_complete_mask).shift(1).ffill()
    asian_low_final = daily_asian_low.where(asian_complete_mask).shift(1).ffill()
    asian_range = (asian_high_final - asian_low_final) / c
    
    feat['asian_range'] = asian_range
    feat['dist_asian_high'] = (c - asian_high_final) / c
    feat['dist_asian_low'] = (c - asian_low_final) / c
    
    # ── Hours into current session ─────────────────────────────────
    session_starts = pd.Series(0, index=df_1h.index, dtype=int)
    session_starts[hours == 0] = 1
    session_starts[hours == 8] = 1
    session_starts[hours == 13] = 1
    session_id = session_starts.cumsum()
    feat['hours_into_session'] = session_id.groupby(session_id).cumcount()
    
    # ── Previous session range ─────────────────────────────────────
    # Range of the previous complete session (normalized)
    session_high = h.groupby(session_id).transform('max')
    session_low = l.groupby(session_id).transform('min')
    session_range = session_high - session_low
    
    # Get final range per session, shift so current session sees previous
    session_final_range = session_range.groupby(session_id).last()
    prev_session_range = session_final_range.shift(1)
    feat['prev_session_range'] = session_id.map(prev_session_range) / c
    
    return feat


print('Session feature function ready. (all session data shifted to prevent lookahead)')

In [ ]:
def compute_crossTF_features(feat_1h: pd.DataFrame, feat_4h: pd.DataFrame, feat_1d: pd.DataFrame) -> pd.DataFrame:
    """
    Compute cross-timeframe alignment features.
    These capture confluence — when multiple timeframes agree on direction.
    """
    feat = pd.DataFrame(index=feat_1h.index)
    
    # Trend alignment — are 1H and 4H moving averages in same direction?
    if 'ema_cross_1H' in feat_1h.columns and 'ema_cross_4H' in feat_4h.columns:
        feat['trend_align_1h_4h'] = (feat_1h['ema_cross_1H'] == feat_4h['ema_cross_4H']).astype(int)
    
    if 'ema_cross_4H' in feat_4h.columns and 'ema_cross_1D' in feat_1d.columns:
        feat['trend_align_4h_1d'] = (feat_4h['ema_cross_4H'] == feat_1d['ema_cross_1D']).astype(int)
    
    # Full confluence — all three timeframes agree
    if all(c in feat.columns for c in ['trend_align_1h_4h', 'trend_align_4h_1d']):
        feat['full_confluence'] = (feat['trend_align_1h_4h'] & feat['trend_align_4h_1d']).astype(int)
    
    # Volatility regime — is short-term vol expanding vs long-term?
    if 'atr_ratio_1H' in feat_1h.columns and 'atr_ratio_1D' in feat_1d.columns:
        feat['vol_expansion'] = (feat_1h['atr_ratio_1H'] > feat_1d['atr_ratio_1D']).astype(int)
    
    # RSI alignment across timeframes
    if 'rsi_14_1H' in feat_1h.columns and 'rsi_14_4H' in feat_4h.columns:
        feat['rsi_align_1h_4h'] = np.sign(feat_1h['rsi_14_1H'] - 50) == np.sign(feat_4h['rsi_14_4H'] - 50)
        feat['rsi_align_1h_4h'] = feat['rsi_align_1h_4h'].astype(int)
    
    # Momentum confluence score — how many timeframes show bullish momentum?
    scores = []
    for col, df_ in [('rsi_14_1H', feat_1h), ('rsi_14_4H', feat_4h), ('rsi_14_1D', feat_1d)]:
        if col in df_.columns:
            scores.append((df_[col] > 50).astype(int))
    if scores:
        feat['momentum_confluence'] = sum(scores)
    
    return feat


print('Cross-TF feature function ready.')

In [ ]:
def build_feature_matrix(pair: str) -> pd.DataFrame:
    """
    Build the full feature matrix for a given pair.
    Base index = 1H candles.
    Features from 5m, 15m, 1H, 4H, 1D — all aligned to 1H timestamps.
    
    IMPORTANT: 4H and 1D features are shift(1)'d before ffill to prevent lookahead.
    At time T, the model sees the PREVIOUS completed 4H/1D bar, not the current one.
    """
    print(f'  Loading data...')
    
    # Load all timeframes
    dfs = {}
    for tf in ['5m', '15m', '1H', '4H', '1D']:
        path = PROCESSED_DIR / f'{pair}_{tf}.parquet'
        if path.exists():
            dfs[tf] = pd.read_parquet(path)
        else:
            print(f'  WARNING: Missing {tf} data for {pair}')
    
    # Base index is 1H
    base = dfs['1H'].copy()
    
    print(f'  Computing features...')
    
    # Compute features for each timeframe
    feat_5m  = compute_features(dfs['5m'],  '5m')  if '5m'  in dfs else None
    feat_15m = compute_features(dfs['15m'], '15m') if '15m' in dfs else None
    feat_1h  = compute_features(dfs['1H'],  '1H')
    feat_4h  = compute_features(dfs['4H'],  '4H')  if '4H'  in dfs else None
    feat_1d  = compute_features(dfs['1D'],  '1D')  if '1D'  in dfs else None
    
    # Time features (based on 1H index)
    feat_time = compute_time_features(base)
    
    # Session features (based on 1H OHLCV)
    feat_session = compute_session_features(base)
    
    print(f'  Aligning timeframes to 1H index...')
    
    # Start with 1H features (same index)
    all_features = feat_1h.copy()
    
    # Merge 5m features — resample to 1H using last value (no lookahead)
    if feat_5m is not None:
        feat_5m_1h = feat_5m.resample('1h').last()
        all_features = all_features.join(feat_5m_1h, how='left')
    
    # Merge 15m features — resample to 1H
    if feat_15m is not None:
        feat_15m_1h = feat_15m.resample('1h').last()
        all_features = all_features.join(feat_15m_1h, how='left')
    
    # Merge 4H features — shift(1) THEN forward fill to prevent lookahead
    # At time T, you see the PREVIOUS completed 4H bar, not the current partial one
    if feat_4h is not None:
        feat_4h_shifted = feat_4h.shift(1)
        feat_4h_1h = feat_4h_shifted.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_4h_1h, how='left')
    
    # Merge 1D features — shift(1) THEN forward fill
    if feat_1d is not None:
        feat_1d_shifted = feat_1d.shift(1)
        feat_1d_1h = feat_1d_shifted.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_1d_1h, how='left')
    
    # Cross-TF features (using shifted 4H/1D)
    if feat_4h is not None and feat_1d is not None:
        feat_4h_aligned = feat_4h.shift(1).reindex(all_features.index, method='ffill')
        feat_1d_aligned = feat_1d.shift(1).reindex(all_features.index, method='ffill')
        feat_cross = compute_crossTF_features(feat_1h, feat_4h_aligned, feat_1d_aligned)
        all_features = all_features.join(feat_cross, how='left')
    
    # Time features
    all_features = all_features.join(feat_time, how='left')
    
    # Session features
    all_features = all_features.join(feat_session, how='left')
    
    # Pair identity — encoded as integer
    pair_map = {p: i for i, p in enumerate(PAIRS)}
    all_features['pair_id'] = pair_map[pair]
    
    print(f'  Feature matrix shape: {all_features.shape}')
    
    return all_features


print('Feature matrix builder ready. (4H/1D shifted to prevent lookahead)')

In [ ]:
# Build and save feature matrix for each pair
for pair in PAIRS:
    print(f'\nBuilding features for {pair}...')
    
    try:
        feat_df = build_feature_matrix(pair)
        out_path = FEATURES_DIR / f'{pair}_features.parquet'
        feat_df.to_parquet(out_path)
        print(f'  Saved: {out_path.name} — {feat_df.shape[0]} rows × {feat_df.shape[1]} features')
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()

print('\nAll pairs done!')

## 2b. Cross-Pair Correlation Features

In [ ]:
# ## 2b. Cross-Pair Correlation Features
# Rolling correlation between each pair and all others.
# Captures regime shifts — during geopolitical stress or risk-off events,
# normally uncorrelated pairs suddenly move together.

print('Computing cross-pair correlations...')

# Load 1H close prices for all pairs
closes = {}
for pair in PAIRS:
    df_1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    closes[pair] = df_1h['close']

close_df = pd.DataFrame(closes)

# Compute rolling correlations at two windows
# 24H = short-term correlation regime
# 168H = weekly correlation regime
for window, w_name in [(24, '24H'), (168, '1W')]:
    print(f'  Computing {w_name} rolling correlations...')
    rolling_corr = close_df.rolling(window).corr()
    
    for pair in PAIRS:
        feat_path = FEATURES_DIR / f'{pair}_features.parquet'
        pair_data = pd.read_parquet(feat_path)
        
        for other_pair in PAIRS:
            if other_pair == pair:
                continue
            col_name = f'corr_{other_pair}_{w_name}'
            try:
                corr_series = rolling_corr.xs(pair, level=1)[other_pair]
                pair_data[col_name] = corr_series.reindex(pair_data.index, method='ffill')
            except Exception as e:
                pair_data[col_name] = np.nan
        
        pair_data.to_parquet(feat_path)
    
    print(f'  {w_name} correlations added to all pairs.')

# Verify
sample = pd.read_parquet(FEATURES_DIR / 'EURUSD_features.parquet')
corr_cols = [c for c in sample.columns if c.startswith('corr_')]
print(f'\nCorrelation features added: {len(corr_cols)}')
print(f'Example columns: {corr_cols[:4]}')
print('Cross-pair correlations complete.')


## 2c. Aggregated Correlation Features + PC1 Share


In [ ]:
# Aggregated correlation features + PC1 share
# 1. corr_mean_abs_{W} — average absolute correlation vs all other pairs (connectivity)
# 2. corr_std_{W}      — std of correlations (stability)
# 3. pc1_share_{W}     — variance explained by first PC of 7x7 correlation matrix
#                        High = one factor dominates (crisis/risk-off)
#                        Low  = pairs move independently (normal market)

from sklearn.decomposition import PCA

print('Computing aggregated correlation features + PC1 share...')

for window, w_name in [(24, '24H'), (168, '1W')]:
    print(f'\n  Window: {w_name}')
    rolling_corr = close_df.rolling(window).corr()

    # Compute PC1 share per timestamp
    # Sample every 4H then forward fill — correlation matrix changes slowly
    timestamps = close_df.index[window:]
    sample_ts  = timestamps[::4]
    pc1_series = pd.Series(index=close_df.index, dtype=float)

    print(f'  Computing PCA on {len(sample_ts)} sampled timestamps...')
    for ts in sample_ts:
        try:
            corr_mat = rolling_corr.xs(ts, level=0)
            if corr_mat.isnull().any().any():
                continue
            pca = PCA(n_components=1)
            pca.fit(corr_mat.values)
            pc1_series[ts] = pca.explained_variance_ratio_[0]
        except:
            continue

    pc1_series = pc1_series.ffill()

    # Add features to each pair's file
    for pair in PAIRS:
        feat_path = FEATURES_DIR / f'{pair}_features.parquet'
        pair_data = pd.read_parquet(feat_path)

        other_pairs = [p for p in PAIRS if p != pair]
        raw_corrs   = pair_data[[f'corr_{op}_{w_name}' for op in other_pairs]]

        pair_data[f'corr_mean_abs_{w_name}'] = raw_corrs.abs().mean(axis=1)
        pair_data[f'corr_std_{w_name}']      = raw_corrs.std(axis=1)
        pair_data[f'pc1_share_{w_name}']     = pc1_series.reindex(pair_data.index, method='ffill')

        pair_data.to_parquet(feat_path)

    print(f'  {w_name} aggregated features added to all pairs.')

# Verify
sample = pd.read_parquet(FEATURES_DIR / 'EURUSD_features.parquet')
new_cols = [c for c in sample.columns if 'corr_mean' in c or 'corr_std' in c or 'pc1' in c]
print(f'\nNew aggregated features: {new_cols}')
print('Done.')

## 3. Compute Labels

For each horizon we compute the forward log return.
This is the target variable `μ` our regression models will predict.

**No lookahead:** labels use future candles only. Features use past candles only.

In [ ]:
# Horizon definitions in number of 1H bars
HORIZONS = {
    '1H':  1,
    '4H':  4,
    '1D':  24,
    '7D':  168,
}

def compute_labels(pair: str, feat_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute forward log returns for each horizon.
    Uses 1H close prices.
    """
    df_1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    close = df_1h['close'].reindex(feat_df.index)
    
    labels = pd.DataFrame(index=feat_df.index)
    
    for horizon_name, n_bars in HORIZONS.items():
        # Forward log return — shift(-n) gives us future price
        labels[f'label_{horizon_name}'] = np.log(close.shift(-n_bars) / close)
    
    return labels


print('Label function ready.')
print(f'Horizons: {HORIZONS}')

In [ ]:
# Combine all pairs into one dataset with features + labels
all_dfs = []

for pair in PAIRS:
    print(f'Processing labels for {pair}...')
    
    feat_path = FEATURES_DIR / f'{pair}_features.parquet'
    if not feat_path.exists():
        print(f'  Missing features, skipping.')
        continue
    
    feat_df = pd.read_parquet(feat_path)
    label_df = compute_labels(pair, feat_df)
    
    combined = feat_df.join(label_df, how='left')
    combined['pair'] = pair
    
    all_dfs.append(combined)
    print(f'  {pair}: {combined.shape}')

# Combine all pairs
df_all = pd.concat(all_dfs, axis=0)
df_all = df_all.sort_index()

print(f'\nCombined dataset shape: {df_all.shape}')
print(f'Date range: {df_all.index[0]} -> {df_all.index[-1]}')
print(f'Pairs: {df_all["pair"].unique()}')

In [ ]:
# Drop rows where any label is NaN (end of dataset — no future data)
label_cols = [f'label_{h}' for h in HORIZONS.keys()]
df_all = df_all.dropna(subset=label_cols)

# Save combined dataset
out_path = FEATURES_DIR / 'all_pairs_features_labels.parquet'
df_all.to_parquet(out_path)

print(f'Combined dataset saved: {df_all.shape}')
print(f'Features: {df_all.shape[1] - len(label_cols) - 1} columns')
print(f'Labels: {label_cols}')
print(f'\nLabel statistics:')
df_all[label_cols].describe()

## 4. Validate Features

In [ ]:
# Check for NaN ratios per feature — high NaN = feature may be problematic
feat_cols = [c for c in df_all.columns if c not in label_cols + ['pair']]

nan_ratio = df_all[feat_cols].isnull().mean().sort_values(ascending=False)
print('Features with >10% NaN:')
print(nan_ratio[nan_ratio > 0.1])
print(f'\nTotal features: {len(feat_cols)}')
print(f'Features with 0% NaN: {(nan_ratio == 0).sum()}')

In [ ]:
# Label distribution — check for class balance and outliers
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#080c14')

for ax, horizon in zip(axes.flatten(), HORIZONS.keys()):
    col = f'label_{horizon}'
    data = df_all[col].dropna()
    
    # Clip extreme outliers for visualization
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(p1, p99)
    
    ax.hist(data_clipped, bins=100, color='#4fc3f7', alpha=0.7, edgecolor='none')
    ax.axvline(0, color='#ff4757', linewidth=1.5, linestyle='--')
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white')
    ax.set_title(f'{horizon} Returns', color='white')
    
    pct_up = (data > 0).mean()
    ax.text(0.02, 0.95, f'Up: {pct_up:.1%}  Down: {1-pct_up:.1%}',
            transform=ax.transAxes, color='white', fontsize=9, va='top')
    
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

plt.suptitle('Label Distributions by Horizon', color='white', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation check — flag highly correlated features (>0.95)
# High correlation = redundant features that add noise
sample = df_all[feat_cols].fillna(0).sample(min(10000, len(df_all)))
corr_matrix = sample.corr().abs()

# Find highly correlated pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, upper.loc[row, col]) 
             for col in upper.columns 
             for row in upper.index 
             if upper.loc[row, col] > 0.95]

print(f'Highly correlated feature pairs (>0.95): {len(high_corr)}')
for a, b, v in sorted(high_corr, key=lambda x: -x[2])[:20]:
    print(f'  {a} <-> {b}: {v:.3f}')

In [ ]:
# Remove redundant highly correlated features
# Keep higher timeframe version (more stable), drop lower timeframe duplicate

TF_PRIORITY = ['1D', '4H', '1H', '15m', '5m']

def get_tf(col):
    for tf in TF_PRIORITY:
        if col.endswith(f'_{tf}'):
            return TF_PRIORITY.index(tf)
    return 999

cols_to_drop = set()
for a, b, v in high_corr:
    if get_tf(a) <= get_tf(b):
        cols_to_drop.add(b)
    else:
        cols_to_drop.add(a)

# Fix: keep corr_mean_abs, drop corr_std instead
cols_to_drop.discard('corr_mean_abs_1W')
cols_to_drop.add('corr_std_1W')

# Fix: keep atr_14 at 1H and 4H (most used timeframes), they're not truly redundant
cols_to_drop.discard('atr_14_norm_1H')
cols_to_drop.discard('atr_14_norm_4H')

print(f'Dropping {len(cols_to_drop)} redundant features:')
for c in sorted(cols_to_drop):
    print(f'  {c}')

df_all = df_all.drop(columns=list(cols_to_drop))
df_all.to_parquet(FEATURES_DIR / 'all_pairs_features_labels.parquet')
print(f'\nCleaned dataset shape: {df_all.shape}')

In [ ]:
# Final summary
print('=' * 50)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 50)
print(f'Total rows:     {len(df_all):,}')
print(f'Total features: {len(feat_cols)}')
print(f'Pairs:          {len(PAIRS)}')
print(f'Date range:     {df_all.index[0].date()} -> {df_all.index[-1].date()}')
print(f'\nLabel columns:  {label_cols}')
print(f'\nOutput file:    all_pairs_features_labels.parquet')